# Advanced Reinforcement Learning Project - LunarLander-v3

## Experiment 1
**Task**: Solve "LunarLander-v3".

### Objectives:
1.  **Standard DQN**: Implement DQN, explain design choices, and tune hyperparameters (at least 2).
2.  **Improvement**: Implement Double DQN (DDQN) and compare.
3.  **Analysis**: Discuss performance and test robustness (Wind/Turbulence).


In [ ]:
!pip install gymnasium[box2d] renderlab moviepy

In [ ]:
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from typing import Sequence
from collections import namedtuple, deque
import itertools
import random
import warnings
warnings.filterwarnings("ignore")

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Hyperparameters

In [ ]:
GAMMA = 0.99
BATCH_SIZE = 128
BUFFER_SIZE = 100000
MIN_REPLAY_SIZE = 1000
EPS_START = 0.9
EPS_END = 0.05
EPS_DECAY = 0.995
TARGET_UPDATE_FREQ = 10
LR = 5e-4

In [ ]:
# Initialize Environment
env = gym.make("LunarLander-v3", render_mode="rgb_array")

## Replay Memory & DQN Class

In [ ]:
Transition = namedtuple('Transition', ('states', 'actions', 'rewards', 'dones', 'next_states'))

class Replay_memory():
    def __init__(self, env, fullsize, minsize, batchsize):
        self.env = env
        self.memory = deque(maxlen=fullsize)
        self.rewards = deque(maxlen=50)
        self.batchsize = batchsize
        self.minsize = minsize

    def append(self, transition):
        self.memory.append(transition)

    def sample_batch(self):
        batch = random.sample(self.memory, self.batchsize)
        batch = Transition(*zip(*batch))
        states = torch.from_numpy(np.array(batch.states, dtype=np.float32)).to(device)
        actions = torch.from_numpy(np.array(batch.actions, dtype=np.int64)).unsqueeze(1).to(device)
        rewards = torch.from_numpy(np.array(batch.rewards, dtype=np.float32)).unsqueeze(1).to(device)
        dones = torch.from_numpy(np.array(batch.dones, dtype=bool)).unsqueeze(1).to(device)
        next_states = torch.from_numpy(np.array(batch.next_states, dtype=np.float32)).to(device)
        return states, actions, rewards, dones, next_states

    def initialize(self):
        obs, _ = self.env.reset()
        for _ in range(self.minsize):
            action = self.env.action_space.sample()
            new_obs, reward, terminated, truncated, _ = self.env.step(action)
            done = terminated or truncated
            transition = Transition(obs, action, reward, done, new_obs)
            self.append(transition)
            obs = new_obs
            if done:
                self.env.reset()
        return self

In [ ]:
class DQN(nn.Module):
    def __init__(self, ninputs, noutputs, hidden_size=64):
        super(DQN, self).__init__()
        # Modified to use ReLU and 2 hidden layers for LunarLander complexity
        self.a1 = nn.Linear(ninputs, hidden_size)
        self.a2 = nn.Linear(hidden_size, hidden_size)
        self.a3 = nn.Linear(hidden_size, noutputs)

    def forward(self, X):
        o = self.a1(X)
        o = F.relu(o)
        o = self.a2(o)
        o = F.relu(o)
        o = self.a3(o)
        return o

    def __call__(self, X):
        return self.forward(X)

In [ ]:
def epsilon_greedy_policy(policy_net, epsilon, obs, env):
    rnd_sample = random.random()
    if rnd_sample <= epsilon:
        action = env.action_space.sample()
    else:
        with torch.no_grad():
            # obs needs to be tensor on device
            obs_t = torch.tensor(obs, dtype=torch.float32).unsqueeze(0).to(device)
            action = int(torch.argmax(policy_net(obs_t)))
    return action

## Training Logic
Wrapped in a function to allow running multiple experiments (Part 1 Tuning & Part 2 DDQN).

In [ ]:
def run_training(hyperparams_dict, name="DQN_Run", double_dqn=False, num_episodes=600, seed=42):
    # Unpack hyperparameters
    gamma = hyperparams_dict.get('GAMMA', GAMMA)
    batch_size = hyperparams_dict.get('BATCH_SIZE', BATCH_SIZE)
    buffer_size = hyperparams_dict.get('BUFFER_SIZE', BUFFER_SIZE)
    min_replay_size = hyperparams_dict.get('MIN_REPLAY_SIZE', MIN_REPLAY_SIZE)
    eps_start = hyperparams_dict.get('EPS_START', EPS_START)
    eps_end = hyperparams_dict.get('EPS_END', EPS_END)
    eps_decay = hyperparams_dict.get('EPS_DECAY', EPS_DECAY)
    target_update_freq = hyperparams_dict.get('TARGET_UPDATE_FREQ', TARGET_UPDATE_FREQ)
    lr = hyperparams_dict.get('LR', LR)
    
    # Initialize seeds
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    # Setup
    env = gym.make("LunarLander-v3", render_mode=None)
    replay_memory = Replay_memory(env, buffer_size, min_replay_size, batch_size).initialize()
    
    dqn_policy = DQN(env.observation_space.shape[0], env.action_space.n).to(device)
    dqn_target = DQN(env.observation_space.shape[0], env.action_space.n).to(device)
    dqn_target.load_state_dict(dqn_policy.state_dict())
    dqn_target.eval()

    optimizer = torch.optim.Adam(dqn_policy.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    obs, _ = env.reset(seed=seed)
    eps_threshold = eps_start
    episode = 1
    episode_reward = 0
    
    rewards_history = []

    print(f"[{name}] Starting... DoubleDQN={double_dqn} | LR={lr} | BS={batch_size}")

    for step in itertools.count():
        action = epsilon_greedy_policy(dqn_policy, eps_threshold, obs, env)
        new_obs, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        
        replay_memory.append(Transition(obs, action, reward, done, new_obs))
        episode_reward += reward
        obs = new_obs

        if done:
            replay_memory.rewards.append(episode_reward)
            rewards_history.append(episode_reward)
            
            eps_threshold = max(eps_end, eps_threshold * eps_decay)
            
            obs, _ = env.reset()
            episode_reward = 0
            
            # Check solved condition
            avg_res = np.mean(list(replay_memory.rewards)) # rewards deque has maxlen=50
            
            if episode % 50 == 0:
                print(f'Episode: {episode} Avg Results: {avg_res:.2f} Epsilon: {eps_threshold:.2f}')
            
            if avg_res >= 195.0:
                print(f'Solved at episode: {episode} Avg Results: {avg_res:.2f}')
                break
            
            episode += 1
            if episode > num_episodes:
                break

        # Train step
        b_states, b_actions, b_rewards, b_dones, b_next_states = replay_memory.sample_batch()

        qvalues = dqn_policy(b_states).gather(1, b_actions)

        with torch.no_grad():
            if double_dqn:
                next_actions = dqn_policy(b_next_states).argmax(dim=1, keepdim=True)
                next_qvalues = dqn_target(b_next_states).gather(1, next_actions)
            else:
                next_qvalues = dqn_target(b_next_states).max(1, keepdim=True)[0]
                
            expected_qvalues = b_rewards + GAMMA * (1 - b_dones.type(torch.int64)) * next_qvalues

        loss = loss_fn(qvalues, expected_qvalues)
        optimizer.zero_grad()
        loss.backward()
        # Gradient clipping
        for param in dqn_policy.parameters():
            param.grad.data.clamp_(-1, 1)
        optimizer.step()

        if episode % TARGET_UPDATE_FREQ == 0:
            dqn_target.load_state_dict(dqn_policy.state_dict())
            
    env.close()
    return rewards_history, dqn_policy

## Experiment 1: Standard DQN & Tuning

In [ ]:
# Default params
params_1 = {
    'GAMMA': GAMMA,
    'BATCH_SIZE': 64,
    'LR': 5e-4
}

scores_1, model_1 = run_training(params_1, name="Run1_BS64_LR5e-4", double_dqn=False)

In [ ]:
# Tuned params: Higher Batch Size
params_2 = {
    'GAMMA': GAMMA,
    'BATCH_SIZE': 128,
    'LR': 5e-4
}

scores_2, model_2 = run_training(params_2, name="Run2_BS128_LR5e-4", double_dqn=False)

In [ ]:
# Tuned params: Higher LR
params_3 = {
    'GAMMA': GAMMA,
    'BATCH_SIZE': 64,
    'LR': 1e-3
}

scores_3, model_3 = run_training(params_3, name="Run3_BS64_LR1e-3", double_dqn=False)

## Experiment 2: Double DQN

In [ ]:
# Double DQN with best standard params (assuming standard default for comparison)
scores_ddqn, model_ddqn = run_training(params_1, name="DoubleDQN", double_dqn=True)

## Comparisons

In [ ]:
def plot_comparison(scores_list, labels, title):
    plt.figure(figsize=(10, 6))
    for scores, label in zip(scores_list, labels):
        window = 50
        if len(scores) >= window:
            moving_avg = np.convolve(scores, np.ones(window)/window, mode='valid')
            plt.plot(moving_avg, label=label)
        else:
            plt.plot(scores, label=label)
    plt.xlabel('Episode')
    plt.ylabel('Average Reward (50-ep)')
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()

# Compare Tuning
plot_comparison([scores_1, scores_2, scores_3], 
                ['Base (BS64, LR5e-4)', 'BS128', 'LR1e-3'], 
                'Hyperparameter Tuning (Standard DQN)')

# Compare STD vs DDQN
plot_comparison([scores_1, scores_ddqn], 
                ['Standard DQN', 'Double DQN'], 
                'Standard vs Double DQN')

## Experiment 3: Analysis & Robustness
Check performance under wind and turbulence.

In [ ]:
def test_robustness(agent, wind_powers=[0, 5, 10, 15, 20]):
    avg_rewards = []
    for w in wind_powers:
        # Create env with wind (custom wind_power)
        # Note: Gymnasium's LunarLander allows passing enable_wind=True and wind_power
        env_test = gym.make("LunarLander-v3", enable_wind=True, wind_power=w, turbulence_power=0.5)
        
        total = 0
        for _ in range(20): # 20 test episodes
            obs, _ = env_test.reset()
            done = False
            while not done:
                action = epsilon_greedy_policy(agent, 0.0, obs, env_test) # greedy
                obs, r, term, trunc, _ = env_test.step(action)
                done = term or trunc
                total += r
        avg_rewards.append(total / 20.0)
        env_test.close()
    return avg_rewards

winds = [0.0, 5.0, 10.0, 15.0, 20.0]
print("Testing Robustness (Standard DQN)...")
res_std = test_robustness(model_1, winds)

print("Testing Robustness (Double DQN)...")
res_ddqn = test_robustness(model_ddqn, winds)

plt.figure(figsize=(8, 5))
plt.plot(winds, res_std, marker='o', label='Standard DQN')
plt.plot(winds, res_ddqn, marker='s', label='Double DQN')
plt.xlabel('Wind Power')
plt.ylabel('Avg Reward (20 eps)')
plt.title('Robustness to Wind')
plt.legend()
plt.grid(True)
plt.show()

## Visualization
Visualizing the best performing agent (Double DQN).

In [ ]:
import renderlab as rl

# Use the best model (e.g., DDQN)
best_model = model_ddqn

env = gym.make("LunarLander-v3", render_mode="rgb_array")
env = rl.RenderFrame(env, "./video")

obs, info = env.reset()

while True:
  action = epsilon_greedy_policy(best_model, 0.0, obs, env)
  obs, reward, terminated, truncated, info = env.step(action)

  if terminated or truncated:
    break

env.play()